In [ ]:
import re
import os
import random
import json
from pprint import pprint
from openai import OpenAI
import pandas as pd
from datasets import load_dataset
from goatools import obo_parser
import concurrent.futures
import random
from datasets import concatenate_datasets, DatasetDict
from bioreason2.dataset.utils import format_go_terms_with_names, filter_go_terms_to_leaf_terms
from datetime import datetime

random.seed(42)
client = OpenAI()
go_dag = obo_parser.GODag("bioreason2/dataset/go-basic.obo", optional_attrs={'relationship'})

DATASET_DIR = "data/reasoning/v5"
OUTPUT_DIR = f"{DATASET_DIR}/individual"
BATCH_INPUTS_DIR = f"{DATASET_DIR}/batch_inputs"
BATCH_OUTPUTS_DIR = f"{DATASET_DIR}/batch_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BATCH_INPUTS_DIR, exist_ok=True)
os.makedirs(BATCH_OUTPUTS_DIR, exist_ok=True)

In [ ]:
def ask_gpt(client: OpenAI, message: str = "Hello", reasoning_effort = "low") -> object:
    """Create a chat response using OpenAI's API."""
    response = client.responses.create(
        model="gpt-5",
        reasoning={ "effort": reasoning_effort },
        input=[
            {
                "role": "user",
                "content": message
            }
        ]
    )
    return response

response = ask_gpt(client, "Don't think, just say BioReason2")
print(response.output_text)

In [ ]:
dataset = load_dataset("wanglab/cafa5", name="experiment_data_with_interpro", cache_dir="/large_storage/goodarzilab/bioreason/cache_dir")
dataset = concatenate_datasets([dataset['train'], dataset['validation']])
dataset = dataset.map(lambda x: {
    **x,
    'go_bp_leaf': filter_go_terms_to_leaf_terms(x['go_bp'], go_dag),
    'go_mf_leaf': filter_go_terms_to_leaf_terms(x['go_mf'], go_dag),
    'go_cc_leaf': filter_go_terms_to_leaf_terms(x['go_cc'], go_dag)
})
dataset = dataset.map(lambda x: {
    **x,
    'go_bp_leaf': "Not known" if x['go_bp_leaf'] is None else "\n".join(format_go_terms_with_names(x['go_bp_leaf'], go_dag)),
    'go_mf_leaf': "Not known" if x['go_mf_leaf'] is None else "\n".join(format_go_terms_with_names(x['go_mf_leaf'], go_dag)),
    'go_cc_leaf': "Not known" if x['go_cc_leaf'] is None else "\n".join(format_go_terms_with_names(x['go_cc_leaf'], go_dag)),
    'protein_function': "Not known" if x['protein_function'] is None else x['protein_function'],
})
dataset = dataset.remove_columns(['go_bp', 'go_mf', 'go_cc', 'go_ids', 'interaction_partners', 'interpro_ids', 'length', 'interpro_location', 'string_id', 'structure_path'])

In [ ]:
temporal_dataset = load_dataset("wanglab/cafa5", name="cafa5_temporal_2023_2025", cache_dir="/large_storage/goodarzilab/bioreason/cache_dir")
temporal_dataset = temporal_dataset['test']

In [ ]:
# dataset = dataset.to_pandas()
# temporal_dataset = temporal_dataset.to_pandas()

updated_proteins = list(set(dataset['protein_id']).intersection(set(temporal_dataset['protein_id'])))
new_proteins = list(set(temporal_dataset['protein_id']) - set(dataset['protein_id']))
no_interpro_proteins = list(dataset.filter(lambda x: x['interpro_formatted'] == "")['protein_id'])

print(f"Updated proteins: {len(updated_proteins)}")
print(f"New proteins: {len(new_proteins)}")
print(f"No interpro proteins: {len(no_interpro_proteins)}")

In [ ]:
# # Let's examine an example of an updated protein (one that exists in both datasets)
# example_updated_protein_id = list(updated_proteins)[0]
# print(f"Example updated protein ID: {example_updated_protein_id}")

# # Get the protein data from both datasets
# original_protein = dataset[dataset['protein_id'] == example_updated_protein_id].iloc[0]
# temporal_protein = temporal_dataset[temporal_dataset['protein_id'] == example_updated_protein_id].iloc[0]

# print("\n=== ORIGINAL DATASET ===")
# print(f"Protein ID: {original_protein['protein_id']}")
# print(f"Organism: {original_protein['organism']}")
# print(f"Protein Function: {original_protein['protein_function']}")
# print(f"GO BP: {original_protein['go_bp']}")
# print(f"GO MF: {original_protein['go_mf']}")
# print(f"GO CC: {original_protein['go_cc']}")
# print(f"InterPro: {original_protein['interpro_ids']}")
# print(f"Subcellular Location: {original_protein['subcellular_location']}")

# print("\n=== TEMPORAL DATASET ===")
# print(f"Protein ID: {temporal_protein['protein_id']}")
# print(f"Organism: {temporal_protein['organism']}")
# print(f"Protein Function: {temporal_protein['protein_function']}")
# print(f"GO BP: {temporal_protein['go_bp']}")
# print(f"GO MF: {temporal_protein['go_mf']}")
# print(f"GO CC: {temporal_protein['go_cc']}")
# print(f"InterPro: {temporal_protein['interpro_ids']}")
# print(f"Subcellular Location: {temporal_protein['subcellular_location']}")

# Prompt

In [ ]:
META_PROMPT = """
You are an expert computational biologist excellent at clear scientific communication. Your mission is to analyze detailed protein metadata and generate an exhaustive functional annotation report that simulates the process of genuine scientific discovery.

Your response must consist of two distinct, detailed parts:
1) A "Reasoning Trace": A fluid, concise narrative that demonstrates the step-by-step deductive process a scientist would follow. You must meticulously analyze each piece of evidence, starting from sequence-level features (InterPro domains), and logically build a comprehensive case for the protein's molecular function, biological process, and cellular location before hypothesizing its mechanism.
2) A "Final Summary": A structured summary of the protein's function, following the exact format specified below.

---

### Core Principles of Your Analysis

1. Simulate Abductive Reasoning:
Write from the perspective of inferring function for the first time. Build from foundational sequence-derived facts to higher-level conclusions. Do not use language that implies you are merely confirming pre-known answers (avoid phrases like "provided data", "the interaction list", "the data confirms", etc.). Maintain a discovery-oriented tone. To achieve this, you must treat the provided GO terms and the Subcellular Location not as facts to be validated, but as formal labels for concepts you deduce independently. Furthermore, your reasoning must use causal language (e.g., 'this architecture causes X activity') rather than comparative language.

2. Follow a Strict and Detailed Chain of Inference:
You must use every piece of provided input data within the reasoning trace. Begin with InterPro domains (include IDs, names, and residue spans; reason about their order and architecture). From this analysis, infer Molecular Function (GO MF), then connect to Biological Process (GO BP), and finally to Cellular Component (GO CC) and subcellular location. Then narrate a concise hypothesis to deduce the interaction partners and function of the protein from your reasoning. Your first sentnece should always start by exploring the InterPro domains and their architecture.

3. Create a Self-Contained Logical Proof:
The ultimate goal is to generate a reasoning trace that can stand alone as a complete, logical argument. A reader should be able to follow your deductive path from sequence features to a mechanistic model without ever seeing the original input data. This is why it is critical to build the case step-by-step, to predict and hypothesize outcomes rather than referencing or justifying given facts, and to cohesively weave every piece of evidence into the final narrative. Refer to the examples below to learn how the Interaction Partners are still hypothesized even when given.

---

### Output Format Requirements

• Enclosing Tags: Produce exactly two top-level sections enclosed by these tags:
  <|REASONING|> ... <|/REASONING|>
  <|FINAL_ANSWER|> ... <|/FINAL_ANSWER|>

• Final Summary Structure: Inside <|FINAL_ANSWER|>, output a bullet list with these exact keys (plain text keys, not bolded):
  - Functional Summary
  - InterPro
  - Molecular Function
  - Biological Process
  - Cellular Component
  - Hypothesized Interaction Partners

• Functional Summary:
  A detailed paragraph that summarizes the protein’s function. It should mention all information present in the input Protein Function and describe what the protein does, how it contributes to broader processes, where it is likely located, and the essence of its proposed mechanism. Do not include InterPro IDs, GO terms, residue spans, or other technical labels here. those belong only in the bullet list sections. Do not refer to protein names or other identifiers in the functional summary.

• Completeness:
  - Mention every provided InterPro item (with ID, name, and residue range) in BOTH the reasoning and the "InterPro" bullet list.
  - Include all provided GO leaves in BOTH the reasoning and the corresponding GO bullets.
  - If a GO aspect is missing, infer the most specific GO terms based on the provided data.
  - If protein function is missing or is incomplete, infer the protein function based on the provided data.

• Styling Constraints:
  - Do not bold anything in your output.
  - Keep scientific names and identifiers precise and unambiguous.
  - Keep narrative rigorous and technically specific, with clear causal links.
  - Do not use bullet points in the reasoning trace.

---

**Example 1**

[EXAMPLE INPUT DATA]
- Protein Function: With MSL1, acts as a negative regulator of salt tolerance.
- Organism: Saccharomyces cerevisiae (strain ATCC 204508 / S288c) (Baker's yeast)
- Subcellular Location: Cytoplasm
- InterPro Domains:
    - IPR025279: Stress response protein NST1 (family) [6-160]
    - IPR051195: Fungal stress response NST1 (family) [13-1134]
- GO MF Leaf: GO:0005515 protein binding
- GO BP Leaf: GO:0009651 response to salt stress
- GO CC Leaf: GO:0005737 cytoplasm
- Interaction Partners:
    - AP-2 complex subunit mu
    - Protein CAF40
    - General negative regulator of transcription subunit 3
    - Putative uncharacterized protein YNL089C
    - Uncharacterized protein YKL075C
    - Tubulin beta chain

[EXAMPLE OUTPUT]
<|REASONING|>
IPR025279 (Stress response protein NST1 family) is found in the N-terminal region from residues 6–160, and this domain is fully embedded within IPR051195 (Fungal stress response NST1 family), which spans nearly the entire polypeptide from residues 13–1134. The dominance of broad stress-response family signatures and the absence of catalytic motifs indicate that this protein is not an enzyme but instead acts as a large scaffold specialized for stress adaptation. Such scaffolds typically operate through multivalent contacts with other proteins, supporting GO:0005515 protein binding as the molecular function.

Stress-adaptation scaffolds organize complexes that regulate how cells handle osmotic challenges. The NST family architecture is consistent with assembling regulatory modules that restrain pathways enhancing tolerance to high salt. This architecture is suited to restrain pathways enhancing tolerance to high salt, which corresponds to the process formalized as GO:0009651 response to salt stress.

The domain architecture shows no secretion signals or transmembrane helices, which supports a soluble localization. Proteins with broad scaffolding roles of this kind usually operate in the cytoplasm, where they can access both transcriptional regulators and cytoskeletal machinery. I therefore hypothesize localization to the GO:0005737 cytoplasm.

From this cytoplasmic platform, the protein can assemble inhibitory complexes that down-modulate salt-tolerance outputs. It may recruit transcriptional repressors such as Protein CAF40 and the General negative regulator of transcription subunit 3 to silence tolerance-promoting transcripts, while also engaging AP-2 complex subunit mu and Tubulin beta chain to constrain trafficking and cytoskeletal remodeling. The uncharacterized proteins YNL089C and YKL075C plausibly act as adaptors that stabilize these complexes. Cooperative function with MSL1 could reinforce the inhibitory hub that suppresses salt tolerance.
<|FINAL_ANSWER|>
- Functional Summary: A large cytoplasmic scaffold in baker’s yeast that uses multivalent protein binding to assemble inhibitory complexes during salt stress. It likely suppresses tolerance-promoting transcripts while constraining trafficking and cytoskeletal remodeling, with uncharacterized adaptors stabilizing the assemblies and cooperative action with MSL1 reinforcing negative control.
- InterPro:
    - IPR025279: Stress response protein NST1 (family) [6-160]
    - IPR051195: Fungal stress response NST1 (family) [13-1134]
- Molecular Function:
    - GO:0005515 protein binding
- Biological Process:
    - GO:0009651 response to salt stress
- Cellular Component:
    - GO:0005737 cytoplasm
- Hypothesized Interaction Partners:
    - AP-2 complex subunit mu
    - Protein CAF40
    - General negative regulator of transcription subunit 3
    - Putative uncharacterized protein YNL089C
    - Uncharacterized protein YKL075C
    - Tubulin beta chain
<|/FINAL_ANSWER|>

---

**Example 2**

[EXAMPLE INPUT DATA]
- Protein Function: Ferric-chelate reductases reduce Fe(3+) to Fe(2+) before its transport from the endosome to the cytoplasm.
- Organism: Mus musculus (Mouse)
- Subcellular Location: Membrane; Multi-pass membrane protein
- InterPro Domains:
    - IPR005018: DOMON domain (domain) [216-331]
    - IPR002861: Reeler domain (domain) [13-179]
    - IPR042307: Reeler domain superfamily (homologous_superfamily) [28-158]
    - IPR051237: Ferric-chelate Reductase and Defense (family) [9-272]
    - IPR006593: Cytochrome b561/ferric reductase transmembrane (domain) [335-534]
- GO MF Leaf: GO:0016722 oxidoreductase activity, acting on metal ions
- GO BP Leaf: GO:0006879 intracellular iron ion homeostasis
- GO CC Leaf:
- Interaction Partners:
    - Large ribosomal subunit protein uL23m
    - Large ribosomal subunit protein mL46
    - Large ribosomal subunit protein uL3m
    - Large ribosomal subunit protein uL16m
    - Large ribosomal subunit protein uL29m
    - Large ribosomal subunit protein uL22m
    - Large ribosomal subunit protein bL19m
    - Large ribosomal subunit protein uL13m
    - Large ribosomal subunit protein bL27m
    - Small ribosomal subunit protein uS15m

[EXAMPLE OUTPUT]
<|REASONING|>
The N-terminal region contains IPR002861 (Reeler domain, residues 13–179) within IPR042307 (Reeler domain superfamily, residues 28–158), followed by IPR005018 (DOMON domain, residues 216–331). These domains often form lumen-facing binding modules that capture ferric complexes or small cofactors at the non-cytosolic side of a membrane. Downstream, the C-terminal region contains IPR006593 (Cytochrome b561/ferric reductase transmembrane, residues 335–534), a heme-based multi-pass module that conducts electrons across the lipid bilayer. Inclusion in IPR051237 (Ferric-chelate Reductase and Defense family, residues 9–272) specifies the chemistry as ferric reduction. Taken together, this ordered layout of substrate-binding domains leading into a transmembrane redox core defines GO:0016722 oxidoreductase activity, acting on metal ions, specifically Fe(3+) to Fe(2+) conversion.

Ferric reduction is essential for mobilizing iron into cellular metabolism. This role situates the protein in GO:0006879 intracellular iron ion homeostasis, ensuring reduced iron is available for biosynthetic use. The lumen-facing binding sites and the electron-transfer core indicate that the protein resides in a membrane, functioning as GO:0016021 integral component of membrane. Because iron reduction typically occurs after endocytosis, the most plausible site is the GO:0010008 endosome membrane.

A plausible mechanism is iron channeling. The protein may form transient associations with ribosomal subunits such as uL23m, mL46, uL3m, uL16m, uL29m, uL22m, bL19m, uL13m, bL27m, and uS15m. These contacts would allow newly produced Fe(2+) to be delivered directly to ribosomal assembly pathways, minimizing cytosolic exposure.
<|/REASONING|>
<|FINAL_ANSWER|>
- Functional Summary: A membrane ferric-chelate reductase in mouse that binds ferric complexes on the lumenal side and transfers electrons through a cytochrome b561 core to generate ferrous iron for downstream transport and utilization at the endosomal interface, with transient assemblies likely channeling iron toward intensive biosynthetic pathways while limiting reactive iron exposure.
- InterPro:
    - IPR005018: DOMON domain (domain) [216-331]
    - IPR002861: Reeler domain (domain) [13-179]
    - IPR042307: Reeler domain superfamily (homologous_superfamily) [28-158]
    - IPR051237: Ferric-chelate Reductase and Defense (family) [9-272]
    - IPR006593: Cytochrome b561/ferric reductase transmembrane (domain) [335-534]
- Molecular Function:
    - GO:0016722 oxidoreductase activity, acting on metal ions
- Biological Process:
    - GO:0006879 intracellular iron ion homeostasis
- Cellular Component:
    - GO:0016021 integral component of membrane
    - GO:0010008 endosome membrane
- Hypothesized Interaction Partners:
    - Large ribosomal subunit protein uL23m
    - Large ribosomal subunit protein mL46
    - Large ribosomal subunit protein uL3m
    - Large ribosomal subunit protein uL16m
    - Large ribosomal subunit protein uL29m
    - Large ribosomal subunit protein uL22m
    - Large ribosomal subunit protein bL19m
    - Large ribosomal subunit protein uL13m
    - Large ribosomal subunit protein bL27m
    - Small ribosomal subunit protein uS15m
<|/FINAL_ANSWER|>

---

### New Task: Generate a response for the following protein

[INPUT DATA]
- Protein Function: {protein_function}
- Organism: {organism}
- Subcellular Location: {subcellular_location}
- InterPro Domains: {interpro_formatted}
- GO MF Leaf: {go_mf_leaf}
- GO BP Leaf: {go_bp_leaf}
- GO CC Leaf: {go_cc_leaf}
- Interaction Partners: {ppi_formatted}
""".strip()


In [ ]:
def run_sample(sample):
    prompt = META_PROMPT.format(
        organism=sample["organism"],
        subcellular_location=sample["subcellular_location"],
        interpro_formatted=sample["interpro_formatted"],
        ppi_formatted=sample["ppi_formatted"],
        go_mf_leaf=sample["go_mf_leaf"],
        go_bp_leaf=sample["go_bp_leaf"],
        go_cc_leaf=sample["go_cc_leaf"],
        protein_function=sample["protein_names"] + ". " + sample["protein_function"]
    )

    response = ask_gpt(client, prompt, reasoning_effort="low")
    return response.output_text

# Sample 12

In [ ]:
sample = dataset[12]
pprint(sample)

In [ ]:
print(run_sample(sample))

# Sample 5

In [ ]:
sample = dataset[5]
pprint(sample)

In [ ]:
print(run_sample(sample))

# Sample 100

In [ ]:
sample = dataset[100]
pprint(sample)

In [ ]:
print(run_sample(sample))

# Sample 1000

In [ ]:
sample = dataset[1000]
pprint(sample)

In [ ]:
print(run_sample(sample))

# Sample 10000

In [ ]:
sample = dataset[10000]
pprint(sample)

In [ ]:
print(run_sample(sample))

# Big Run

In [ ]:
# Filter out already processed samples
existing_files = set(os.listdir(OUTPUT_DIR))
samples_to_process = [s for s in dataset if f"{s['protein_id']}.txt" not in existing_files]
print(f"Processing {len(samples_to_process)}/{len(dataset)} samples (skipping existing)")

In [ ]:
# Drop the ones already processed
batch_files = [f for f in os.listdir(BATCH_OUTPUTS_DIR) if f.endswith('.jsonl')]
processed_batch_ids = set()

for batch_file in batch_files:
    with open(os.path.join(BATCH_OUTPUTS_DIR, batch_file), 'r') as f:
        for line in f:
            batch_data = json.loads(line)
            if 'custom_id' in batch_data:
                processed_batch_ids.add(batch_data['custom_id'])

# Filter out samples that were already processed in batch
samples_to_process = [s for s in samples_to_process if s['protein_id'] not in processed_batch_ids]
print(f"Processing {len(samples_to_process)}/{len(dataset)} samples (skipping batch processed)")

In [ ]:
# Filter out proteins without InterPro domains
samples_to_process = [s for s in samples_to_process if s['interpro_formatted']]
print(f"Processing {len(samples_to_process)}/{len(dataset)} samples (skipping existing)")

In [ ]:
# Filter out proteins that got updated
samples_to_process = [s for s in samples_to_process if s['protein_id'] not in updated_proteins]
print(f"Processing {len(samples_to_process)}/{len(dataset)} samples (skipping existing)")

In [ ]:
# Shuffle the samples with seed 23
random.seed(23)
random.shuffle(samples_to_process)
print(f"Processing {len(samples_to_process)}/{len(dataset)} samples")

In [ ]:
def process_sample(sample):
    """Process a single sample and save the result to a file"""
    try:
        result = run_sample(sample)
        protein_id = sample['protein_id']
        filepath = os.path.join(OUTPUT_DIR, f"{protein_id}.txt")

        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(str(result))

        print(f"Processed: {protein_id}")
        return protein_id, "success"

    except Exception as e:
        print(f"Error processing {sample.get('protein_id', 'unknown')}: {str(e)}")
        return sample.get('protein_id', 'unknown'), f"error: {str(e)}"

if samples_to_process:
    with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
        future_to_sample = {executor.submit(process_sample, sample): sample for sample in samples_to_process}
        
        completed = 0
        total = len(samples_to_process)
        
        for future in concurrent.futures.as_completed(future_to_sample):
            protein_id, status = future.result()
            completed += 1
            
            if completed % 100 == 0 or completed == total:
                print(f"Progress: {completed}/{total}")

print("Processing completed!")

# Batch API

In [ ]:
# Function to create batch request for a single sample
def create_batch_request(sample):
    prompt = META_PROMPT.format(
        organism=sample["organism"],
        subcellular_location=sample["subcellular_location"],
        interpro_formatted=sample["interpro_formatted"],
        ppi_formatted=sample["ppi_formatted"],
        go_mf_leaf=sample["go_mf_leaf"],
        go_bp_leaf=sample["go_bp_leaf"],
        go_cc_leaf=sample["go_cc_leaf"],
        protein_function=sample["protein_names"] + ". " + sample["protein_function"]
    )
    
    return {
        "custom_id": sample['protein_id'],
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": "gpt-5",
            "reasoning": {"effort": "low"},
            "input": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    }


# Batch processing parameters
batch_size = 10_000
total_samples = len(samples_to_process)
num_batches = (total_samples + batch_size - 1) // batch_size

print(f"Preparing {num_batches} batches with up to {batch_size} samples each for {total_samples} total samples...")

for i, batch_num in enumerate(range(num_batches)):
    batch_start = batch_num * batch_size
    batch_end = min(batch_start + batch_size, total_samples)
    
    batch_requests = []
    
    print(f"Preparing batch {batch_num + 1}/{num_batches}: samples {batch_start} to {batch_end - 1}")
    
    for i in range(batch_start, batch_end):
        sample = samples_to_process[i]
        batch_request = create_batch_request(sample)
        batch_requests.append(batch_request)
    
    # Get current timestamp for filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Write batch requests to JSONL file in batch_inputs directory
    batch_filename = f"{BATCH_INPUTS_DIR}/cafa5_batch_{timestamp}.jsonl"
    with open(batch_filename, 'w') as f:
        for request in batch_requests:
            f.write(json.dumps(request) + '\n')
    
    print(f"Batch file created: {batch_filename}")
    print(f"Total requests in this batch: {len(batch_requests)}")
    
    # Upload the batch file
    print("Uploading batch file to OpenAI...")
    batch_input_file = client.files.create(
        file=open(batch_filename, "rb"),
        purpose="batch"
    )

    print(f"File uploaded successfully. File ID: {batch_input_file.id}")

    # Create the batch
    print("Creating batch...")
    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "description": f"cafa5_batch_{timestamp}",
            "num_samples": str(batch_end - batch_start)
        }
    )

    print("Batch created successfully!")
    print(f"Batch ID: {batch.id}")
    print(f"Status: {batch.status}")
    print(f"Created at: {batch.created_at}")
    print(f"Input file ID: {batch.input_file_id}")
    print(f"Completion window: {batch.completion_window}")

    # Rename the uploaded file to include batch ID for easier tracking
    renamed_filename = f"{BATCH_INPUTS_DIR}/{batch.id}_cafa5_batch_{timestamp}.jsonl"
    os.rename(batch_filename, renamed_filename)
    print(f"Batch input file renamed to: {renamed_filename}")
    print("-" * 50)

In [ ]:
# List all batches with better formatting
print("Listing recent batches...")
batches = client.batches.list(limit=20)
for batch in batches.data:
    print(f"Batch ID: {batch.id} | Status: {batch.status} | Created: {batch.created_at} | Progress: {batch.request_counts.completed}/{batch.request_counts.total} completed | Failed: {batch.request_counts.failed}")

In [ ]:
# Check all recent batches
print("Checking batch statuses...")
batches = client.batches.list(limit=20)

for batch in batches.data:
    print(f"\nBatch ID: {batch.id} - Status: {batch.status}")
    
    if batch.status == "completed":
        # Get and save results
        file_response = client.files.content(batch.output_file_id)
        output_filename = f"{BATCH_OUTPUTS_DIR}/batch_{batch.id}_{batch.metadata['description'].replace(' ', '_').lower()}.jsonl"
        
        with open(output_filename, 'w') as f:
            f.write(file_response.text)
        
        results_count = len(file_response.text.strip().split('\n'))
        print(f"Results saved: {output_filename} ({results_count} results)")
    
    elif batch.status == "failed":
        print(f"Batch failed: {batch.errors}")

# Push to HuggingFace

In [ ]:
dataset = load_dataset("wanglab/cafa5", name="experiment_data_with_interpro", cache_dir="/large_storage/goodarzilab/bioreason/cache_dir")
dataset = dataset.map(lambda x: {
    **x,
    'go_bp_leaf': filter_go_terms_to_leaf_terms(x['go_bp'], go_dag),
    'go_mf_leaf': filter_go_terms_to_leaf_terms(x['go_mf'], go_dag),
    'go_cc_leaf': filter_go_terms_to_leaf_terms(x['go_cc'], go_dag)
})
dataset = dataset.map(lambda x: {
    **x,
    'go_bp_leaf': "Not known" if x['go_bp_leaf'] is None else "\n".join(format_go_terms_with_names(x['go_bp_leaf'], go_dag)),
    'go_mf_leaf': "Not known" if x['go_mf_leaf'] is None else "\n".join(format_go_terms_with_names(x['go_mf_leaf'], go_dag)),
    'go_cc_leaf': "Not known" if x['go_cc_leaf'] is None else "\n".join(format_go_terms_with_names(x['go_cc_leaf'], go_dag)),
    'protein_function': "Not known" if x['protein_function'] is None else x['protein_function'],
})

In [ ]:
# Load all batch outputs
batch_outputs = {}
output_files = []
duplicates = []

if os.path.exists(BATCH_OUTPUTS_DIR):
    for filename in os.listdir(BATCH_OUTPUTS_DIR):
        if filename.endswith('.jsonl'):
            filepath = os.path.join(BATCH_OUTPUTS_DIR, filename)
            output_files.append(filepath)
            print(f"Found batch output: {filename}")
            
            # Load the JSONL file
            with open(filepath, 'r') as f:
                for line in f:
                    if line.strip():
                        output = json.loads(line.strip())

                        if output['custom_id'] in batch_outputs:
                            duplicates.append(output['custom_id'])
                        else:
                            batch_outputs[output['custom_id']] = output

print(f"Loaded {len(batch_outputs)} batch results from {len(output_files)} files")

In [ ]:
# Load all individual outputs
individual_outputs = {}
for filename in os.listdir(OUTPUT_DIR):
    if filename.endswith('.txt'):
        filepath = os.path.join(OUTPUT_DIR, filename)
        with open(filepath, 'r') as f:
            individual_outputs[filename.replace('.txt', '')] = f.read()

print(f"Loaded {len(individual_outputs)} individual outputs from {len(output_files)} files")

In [ ]:
# Combine all outputs (batch and individual)
all_outputs = {}

# Add batch outputs
for custom_id, output in batch_outputs.items():
    content = output['response']['body']['output'][1]['content'][0]['text']
    all_outputs[custom_id] = content

# Add individual outputs
for custom_id, content in individual_outputs.items():
    all_outputs[custom_id] = content

all_outputs = pd.DataFrame(all_outputs, index=["output"]).T
print(f"Combined total outputs: {len(all_outputs)}")
print(f"Batch outputs: {len(batch_outputs)}")
print(f"Individual outputs: {len(individual_outputs)}")

In [ ]:
# Compile regex patterns with explicit closing tags
reasoning_pattern = re.compile(r'<\|REASONING\|>(.*?)<\|/REASONING\|>', re.DOTALL)
final_answer_pattern = re.compile(r'<\|FINAL_ANSWER\|>(.*?)<\|/FINAL_ANSWER\|>', re.DOTALL)

def extract_content(text):
    reasoning_match = reasoning_pattern.search(text)
    final_answer_match = final_answer_pattern.search(text)
    reasoning = reasoning_match.group(1).strip() if reasoning_match else None
    final_answer = final_answer_match.group(1).strip() if final_answer_match else None
    return reasoning, final_answer

all_outputs[['reasoning', 'final_answer']] = all_outputs['output'].apply(
    lambda x: pd.Series(extract_content(x))
)

In [ ]:
# Drop proteins in all_outputs that have no InterPro domains
no_interpro_proteins = []
for split in dataset.keys():
    split_data = dataset[split].filter(lambda x: x['interpro_formatted'] == "")
    no_interpro_proteins.extend(split_data['protein_id'])

all_outputs = all_outputs[~all_outputs.index.isin(no_interpro_proteins)]
print(f"Total proteins in all_outputs: {len(all_outputs)}")

In [ ]:
# Filter dataset to only include proteins that have outputs, then merge reasoning and final_answer
dataset_with_reasoning = dataset.filter(
    lambda x: x['protein_id'] in all_outputs.index
).map(
    lambda x: {
        **x,
        'reasoning': all_outputs.loc[x['protein_id'], 'reasoning'],
        'final_answer': all_outputs.loc[x['protein_id'], 'final_answer']
    }
)

# Move 5K random datapoints from validation to training with seed 23
random.seed(23)

# Get validation data and select 5K random indices
val_data = dataset_with_reasoning['validation']
val_indices = list(range(len(val_data)))
random.shuffle(val_indices)
move_to_train_indices = val_indices[:5000]
keep_in_val_indices = val_indices[5000:]

# Create new train and validation splits
train_data = dataset_with_reasoning['train']
new_train_data = concatenate_datasets([train_data, val_data.select(move_to_train_indices)])
new_val_data = val_data.select(keep_in_val_indices)

# Create new dataset with updated splits
dataset_with_reasoning = DatasetDict({
    'train': new_train_data,
    'validation': new_val_data
})

dataset_with_reasoning.push_to_hub("wanglab/cafa5", config_name="experiment_data_reasoning")
print(f"Dataset train size: {len(dataset_with_reasoning['train'])}")
print(f"Dataset validation size: {len(dataset_with_reasoning['validation'])}")